# So sánh kết quả phân cụm giao dịch IBFT

Đọc kết quả **K-Means, GMM, Autoencoder + K-Means** trong `output/` để so sánh:
4 bảng (mô hình cuối, kết quả theo K, xếp hạng, ARI/NMI), 4 hình và tra cứu giao dịch theo cụm.
Hình PCA mới đặt ba phương pháp cạnh nhau trên cùng mẫu giao dịch và cùng hệ trục.

K/seed đã được chọn bằng **validation** trong các notebook mô hình.
Notebook này đọc các nhãn và chỉ số đã lưu; PCA chỉ dùng để hiển thị.
Ba chỉ số phân cụm đều thuộc không gian **11 đặc trưng `x_*` gốc**.

**Cách chạy:** chạy xong ba notebook mô hình, rồi *Restart Kernel → Run All*.
Kết quả tổng hợp lưu trong `output/comparison/`.

## 1. Cấu hình

Đổi cỡ mẫu PCA và các lựa chọn tra cứu tại đây. `RUN_ID` cần khớp với ba notebook mô hình.
Lưới K, seed và danh sách đặc trưng được đọc từ cấu hình chung.

In [ ]:
RUN_ID = "run_001"
PCA_SAMPLE_SIZE = 5_000
EVAL_SEED = 42
LOOKUP_METHOD = "kmeans"
LOOKUP_CLUSTER = 0
LOOKUP_SPLIT = "test"       # train, validation, test hoặc all
LOOKUP_LIMIT = 10
MASK_ACCOUNTS = True

METHODS = ["kmeans", "gmm", "autoencoder_kmeans"]
METHOD_NAMES = {
    "kmeans": "K-Means", "gmm": "GMM", "autoencoder_kmeans": "Autoencoder + K-Means",
}
METHOD_COLORS = {"kmeans": "#2457A6", "gmm": "#B27619", "autoencoder_kmeans": "#8B4C70"}
METRICS = ["silhouette", "davies_bouldin", "calinski_harabasz"]
METRIC_NAMES = {
    "silhouette": "Silhouette", "davies_bouldin": "Davies–Bouldin",
    "calinski_harabasz": "Calinski–Harabasz",
}
SPLITS = ["train", "validation", "test"]


## 2. Thư viện và đường dẫn

Đọc `output/config.json`, kết quả của ba phương pháp và dữ liệu trong `output/prepared/`.
Không cần nạp mô hình hoặc PyTorch.

In [ ]:
from pathlib import Path
from itertools import combinations, product
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import Markdown, display
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score


def find_project_root():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "output" / "config.json").is_file() and (folder / "output" / "prepared").is_dir():
            return folder
    raise FileNotFoundError("Chưa có output/config.json hoặc output/prepared; hãy chạy các notebook mô hình trước.")


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def save_figure(fig, filename):
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "output"
PREPARED_DIR = PROJECT_ROOT / "output" / "prepared"
COMPARISON_DIR = OUTPUT_ROOT / "comparison"
FIGURES_DIR = COMPARISON_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})
print("Đầu vào:", OUTPUT_ROOT)
print("Kết quả tổng hợp:", COMPARISON_DIR)


## 3. Đọc kết quả

Dùng cấu hình gọn của ba notebook mô hình hiện tại. Giữ các kiểm tra cần thiết:
cùng cấu hình, đủ lượt quét K/seed, chỉ số hữu hạn và cùng tập giao dịch.
Nhãn được sắp theo **`transaction_id`** trước khi so sánh hoặc tô màu PCA.

In [ ]:
shared_config = read_json(OUTPUT_ROOT / "config.json")
K_GRID = shared_config["k_grid"]
SEEDS = shared_config["seeds"]
FEATURE_COLUMNS = shared_config["feature_columns"]
assert shared_config["run_id"] == RUN_ID, "RUN_ID không khớp cấu hình chung."
assert shared_config["metric_protocol"]["space"] == "prepared_x_features"
assert shared_config["metric_protocol"]["silhouette_validation_and_test"] == "all_rows"

reference = pd.read_parquet(PREPARED_DIR / "model_input.parquet").reset_index(drop=True)
assert reference["transaction_id"].is_unique, "Mã giao dịch bị trùng."
assert np.isfinite(reference[FEATURE_COLUMNS].to_numpy()).all(), "Đặc trưng còn thiếu hoặc vô hạn."
assert reference.groupby("split").size().to_dict() == shared_config["split_row_counts"]
reference_ids = pd.Index(reference["transaction_id"], name="transaction_id")
TEST_MASK = reference["split"].eq("test").to_numpy()
N_TEST = int(TEST_MASK.sum())


In [ ]:
COMMON_KEYS = ["run_id", "k_grid", "seeds", "feature_columns", "split_row_counts", "input_file", "metric_protocol"]
expected_grid = set(product(K_GRID, SEEDS, SPLITS))
results = {}
for method in METHODS:
    folder = OUTPUT_ROOT / method
    config = read_json(folder / "config.json")
    metrics = read_json(folder / "metrics.json")
    sweep = pd.read_csv(folder / "k_sweep.csv")
    labels = pd.read_parquet(folder / "labels.parquet")

    assert all(config[key] == shared_config[key] for key in COMMON_KEYS), f"{method}: khác cấu hình chung."
    assert metrics["method"] == method and metrics["run_id"] == RUN_ID
    assert len(sweep) == len(expected_grid) and set(zip(sweep.k, sweep.seed, sweep.split)) == expected_grid
    assert set(sweep["method"]) == {method} and np.isfinite(sweep[METRICS]).all().all()
    assert sweep["n_eval"].eq(sweep["split"].map(shared_config["split_row_counts"])).all()
    assert labels["transaction_id"].is_unique and set(labels["transaction_id"]) == set(reference_ids)

    # Căn đúng thứ tự giao dịch trước khi ghép các cách chia cụm.
    labels = labels.set_index("transaction_id").loc[reference_ids].reset_index()
    assert labels["split"].equals(reference["split"]), f"{method}: sai tập train/validation/test."
    assert labels["cluster"].between(0, metrics["best_k"] - 1).all(), f"{method}: nhãn cụm không hợp lệ."
    assert metrics["metrics"]["test"]["n_eval"] == N_TEST
    assert np.isfinite([metrics["metrics"]["test"][m] for m in METRICS]).all()
    results[method] = dict(metrics=metrics, sweep=sweep, labels=labels)

print(f"Đã đọc {len(METHODS)} phương pháp; {len(expected_grid)} dòng quét/phương pháp; cùng {N_TEST:,} giao dịch test.")


## 4. Bảng 1 — Mô hình cuối trên test

Mỗi dòng là cấu hình (K, seed) đã được chọn bằng validation trong notebook mô hình.
Bảng lấy nguyên ba chỉ số test từ `metrics.json`; không tính lại hoặc chọn seed bằng test.
Silhouette và Calinski–Harabasz cao hơn tốt hơn; Davies–Bouldin thấp hơn tốt hơn.
Hai cột validation cuối bảng phản ánh trung bình và độ lệch chuẩn giữa các seed ở K được chọn.

In [ ]:
comparison_final = pd.DataFrame([
    {
        "method": method, "run_id": RUN_ID, "split": "test",
        "best_k": results[method]["metrics"]["best_k"],
        "best_seed": results[method]["metrics"]["best_seed"],
        **results[method]["metrics"]["metrics"]["test"],
        "non_empty_clusters": results[method]["metrics"]["non_empty_clusters"]["test"],
        "validation_silhouette_mean": results[method]["metrics"]["best_validation_mean_silhouette"],
        "validation_silhouette_std": results[method]["metrics"]["best_validation_std_silhouette"],
    }
    for method in METHODS
])
final_view = comparison_final.set_index("method").rename(index=METHOD_NAMES)
display(final_view[["best_k", "best_seed", "n_eval", *METRICS,
                    "validation_silhouette_mean", "validation_silhouette_std"]].style.format({
    **{metric: "{:.6f}" for metric in METRICS},
    "validation_silhouette_mean": "{:.6f}", "validation_silhouette_std": "{:.6f}",
    "n_eval": "{:,.0f}",
}))


### So sánh ba chỉ số của mô hình cuối

Mỗi ô có thang đo riêng và trục cột bắt đầu từ 0. Nhãn trên cột là giá trị cụ thể;
không suy luận khác biệt có ý nghĩa thống kê chỉ từ thứ hạng hoặc hình cột.
Nếu K cuối khác nhau, Calinski–Harabasz chỉ được hiển thị tham khảo, không xếp hạng chéo K.

In [ ]:
short_names = ["K-Means", "GMM", "Autoencoder\n+ K-Means"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, metric in zip(axes, METRICS):
    values = comparison_final.set_index("method").loc[METHODS, metric].to_numpy()
    ax.bar(range(len(METHODS)), values, color=[METHOD_COLORS[m] for m in METHODS], width=0.65)
    for position, value in enumerate(values):
        ax.text(position, value, f"{value:.6f}" if metric == "silhouette" else f"{value:.4f}" if metric == "davies_bouldin" else f"{value:.2f}",
                ha="center", va="bottom", fontsize=10)
    direction = "thấp hơn tốt hơn" if metric == "davies_bouldin" else "cao hơn tốt hơn"
    ax.set(title=f"{METRIC_NAMES[metric]}\n({direction})", xticks=range(len(METHODS)),
           xticklabels=short_names, ylabel="Giá trị chỉ số")
    ax.set_ylim(min(0, values.min() * 1.2), max(0.01, values.max()) * 1.22)
    ax.axhline(0, color="#555555", linewidth=0.7)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
fig.suptitle(f"Mô hình cuối đã chọn bằng validation — đánh giá trên {N_TEST:,} giao dịch test")
fig.tight_layout()
save_figure(fig, "comparison_final.png")


## 5. Bảng 2 — Kết quả test theo K, trung bình ± độ lệch chuẩn qua các seed

Mỗi ô là trung bình ± **độ lệch chuẩn mẫu (`ddof=1`)**; đây không phải khoảng tin cậy.
Bảng gồm mọi K, không chọn lại K dựa vào test. Chỉ so sánh Calinski–Harabasz giữa
các phương pháp **ở cùng K**. Với Autoencoder + K-Means, biến động seed bao gồm
khởi tạo cả Autoencoder và K-Means.

In [ ]:
all_sweeps = pd.concat([results[m]["sweep"] for m in METHODS], ignore_index=True)
test_sweep = all_sweeps.loc[all_sweeps["split"].eq("test")].copy()
aggregation = {
    "n_eval": ("n_eval", "first"), "seed_count": ("seed", "nunique"),
    **{f"{metric}_{stat}": (metric, stat) for metric in METRICS for stat in ["mean", "std"]},
}
comparison_by_k = test_sweep.groupby(["method", "k"], sort=False).agg(**aggregation).reset_index()
comparison_by_k.insert(1, "split", "test")
by_k_view = comparison_by_k[["method", "k", "seed_count"]].copy()
by_k_view["method"] = by_k_view["method"].map(METHOD_NAMES)
for metric in METRICS:
    digits = 2 if metric == "calinski_harabasz" else 6
    by_k_view[METRIC_NAMES[metric] + " (mean ± std)"] = [
        f"{mean:.{digits}f} ± {std:.{digits}f}"
        for mean, std in zip(comparison_by_k[f"{metric}_mean"], comparison_by_k[f"{metric}_std"])
    ]
display(by_k_view.set_index(["method", "k"]))


### Ba phương pháp theo K trên cùng tập test

Màu, kiểu nét và marker giữ cố định cho từng phương pháp. Dải màu là ±1 độ lệch chuẩn
qua seed, không biểu thị kiểm định thống kê. Đọc từng vị trí K để so sánh công bằng,
đặc biệt với Calinski–Harabasz.

In [ ]:
line_styles = {"kmeans": ("-", "o"), "gmm": ("--", "s"), "autoencoder_kmeans": ("-.", "^")}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes, METRICS):
    for method in METHODS:
        frame = comparison_by_k.loc[comparison_by_k["method"].eq(method)].set_index("k").reindex(K_GRID)
        mean = frame[f"{metric}_mean"].to_numpy()
        std = frame[f"{metric}_std"].to_numpy()
        linestyle, marker = line_styles[method]
        ax.plot(K_GRID, mean, color=METHOD_COLORS[method], linestyle=linestyle, marker=marker,
                linewidth=1.8, label=METHOD_NAMES[method])
        ax.fill_between(K_GRID, mean - std, mean + std, color=METHOD_COLORS[method], alpha=0.12)
    ax.set(title=METRIC_NAMES[metric], xlabel="Số cụm K", ylabel="Giá trị chỉ số", xticks=K_GRID)
    ax.grid(alpha=0.2)
handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="lower center", ncol=3, frameon=False)
fig.suptitle(f"Test: trung bình ± độ lệch chuẩn qua {len(SEEDS)} seed — so sánh tại cùng K")
fig.tight_layout(rect=(0, 0.08, 1, 0.96))
save_figure(fig, "comparison_by_k.png")


## 6. Bảng 3 — Xếp hạng theo từng chỉ số

Hạng 1 là tốt nhất; bằng đúng giá trị thì nhận hạng trung bình (`method="average"`).
Hạng tổng hợp là trung bình **trọng số bằng nhau** của các hạng, không cộng trực tiếp
ba chỉ số khác thang đo. Đây chỉ là tóm tắt mô tả, không phải xác suất hay kiểm định.

Nếu K cuối khác nhau, bỏ Calinski–Harabasz khỏi xếp hạng tổng hợp và ghi rõ số chỉ số
được dùng. Khi cùng K, dùng đủ ba chỉ số theo đặc tả.
Các cột xếp hạng được lưu chung trong `comparison_final.csv`.

In [ ]:
same_final_k = comparison_final["best_k"].nunique() == 1
ranking_metrics = METRICS if same_final_k else ["silhouette", "davies_bouldin"]
for metric in METRICS:
    comparison_final[f"rank_{metric}"] = (
        comparison_final[metric].rank(method="average", ascending=(metric == "davies_bouldin"))
        if metric in ranking_metrics else np.nan
    )
rank_columns = [f"rank_{m}" for m in ranking_metrics]
comparison_final["mean_rank"] = comparison_final[rank_columns].mean(axis=1)
comparison_final["ranking_metric_count"] = len(ranking_metrics)
comparison_final["overall_rank"] = comparison_final["mean_rank"].rank(method="min", ascending=True).astype(int)
ranking = comparison_final[["method", "best_k", *[f"rank_{m}" for m in METRICS],
                            "mean_rank", "ranking_metric_count", "overall_rank"]].sort_values(["mean_rank", "method"])
display(ranking.set_index("method").rename(index=METHOD_NAMES).style.format({
    **{f"rank_{m}": "{:.1f}" for m in METRICS}, "mean_rank": "{:.3f}",
}, na_rep="Không so sánh khác K"))
if same_final_k:
    print(f"Cả ba mô hình cuối cùng K={int(comparison_final.iloc[0]['best_k'])}; xếp hạng dùng đủ 3 chỉ số.")
else:
    print("K cuối khác nhau: xếp hạng tổng hợp chỉ dùng Silhouette và Davies–Bouldin.")


## 7. Bảng 4 — ARI và NMI giữa các cách chia cụm trên test

Ghép cùng giao dịch theo `transaction_id`, sau đó so từng cặp nhãn của mô hình cuối.
ARI điều chỉnh cho sự trùng hợp ngẫu nhiên (có thể âm); NMI nằm trong [0, 1], ở đây
chuẩn hóa bằng trung bình số học. Cả hai bằng 1 khi hai cách chia cụm trùng nhau,
kể cả khi đổi số thứ tự cụm. **Không có nhãn đúng**, nên ARI/NMI chỉ đo mức độ giống
nhau giữa các phương pháp, không đo độ chính xác hoặc xác định phương pháp tốt nhất.

In [ ]:
test_labels = pd.DataFrame({"transaction_id": reference.loc[TEST_MASK, "transaction_id"].to_numpy()})
for method in METHODS:
    test_labels[method] = results[method]["labels"].loc[TEST_MASK, "cluster"].to_numpy()
agreement_rows = []
for method_a, method_b in combinations(METHODS, 2):
    left, right = test_labels[method_a].to_numpy(), test_labels[method_b].to_numpy()
    agreement_rows.append({
        "method_a": method_a, "method_b": method_b, "split": "test", "n_eval": N_TEST,
        "k_a": results[method_a]["metrics"]["best_k"],
        "k_b": results[method_b]["metrics"]["best_k"],
        "ari": float(adjusted_rand_score(left, right)),
        "nmi": float(normalized_mutual_info_score(left, right, average_method="arithmetic")),
    })
method_agreement = pd.DataFrame(agreement_rows)
agreement_view = method_agreement.replace({"method_a": METHOD_NAMES, "method_b": METHOD_NAMES})
display(agreement_view.style.format({"ari": "{:.6f}", "nmi": "{:.6f}", "n_eval": "{:,.0f}"}))


## 8. Phân bố kích thước cụm của ba mô hình cuối

Cùng một trục tỷ lệ trên test; nhãn cột ghi số lượng và tỷ lệ. Màu biểu thị phương pháp.
**Cụm 0 của hai phương pháp không nhất thiết cùng ý nghĩa**; số thứ tự cụm không được
đối sánh trực tiếp. ARI/NMI phía trên xử lý sự khác nhau của mã cụm.

In [ ]:
size_rows = []
for method in METHODS:
    k = results[method]["metrics"]["best_k"]
    counts = test_labels[method].value_counts().reindex(range(k), fill_value=0)
    for cluster, count in counts.items():
        size_rows.append({"method": method, "cluster": int(cluster), "transaction_count": int(count), "transaction_share": float(count / N_TEST)})
cluster_sizes = pd.DataFrame(size_rows)
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
ymax = min(1.15, float(cluster_sizes["transaction_share"].max()) * 1.25)
for ax, method in zip(axes, METHODS):
    frame = cluster_sizes.loc[cluster_sizes["method"].eq(method)]
    ax.bar(frame["cluster"], frame["transaction_share"], color=METHOD_COLORS[method], width=0.65)
    for row in frame.itertuples():
        ax.text(row.cluster, row.transaction_share, f"{row.transaction_count:,}\n({row.transaction_share:.1%})", ha="center", va="bottom", fontsize=10)
    ax.set(title=METHOD_NAMES[method], xlabel="Mã cụm của phương pháp", xticks=frame["cluster"].tolist(), ylim=(0, ymax))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
axes[0].set_ylabel("Tỷ lệ giao dịch test")
fig.suptitle(f"Phân bố kích thước cụm — cùng {N_TEST:,} giao dịch test")
fig.tight_layout()
save_figure(fig, "cluster_sizes_comparison.png")


## 9. Ba hình PCA trong cùng một khung

Fit **một PCA trên train**, rồi chiếu cùng mẫu giao dịch test lên hai trục PC1–PC2.
Cả ba ô dùng đúng tọa độ, cùng giới hạn trục và cùng thứ tự điểm; chỉ nhãn màu thay đổi.
Đối với Autoencoder + K-Means, PCA vẫn dùng 11 đặc trưng gốc như hai phương pháp còn lại.

Màu cụm giữ như các notebook mô hình. **Cùng màu hoặc cùng mã cụm ở hai phương pháp
không có nghĩa là cùng nhóm giao dịch**; dùng ARI/NMI để đối chiếu cách chia cụm.
PCA chỉ là hình chiếu 2D, không thể hiện toàn bộ cấu trúc 11 chiều.

In [ ]:
pca = PCA(n_components=2, svd_solver="full")
pca.fit(reference.loc[reference["split"] == "train", FEATURE_COLUMNS].to_numpy(dtype=np.float64))
X_test = reference.loc[TEST_MASK, FEATURE_COLUMNS].to_numpy(dtype=np.float64)
plot_rows = np.random.RandomState(EVAL_SEED).permutation(N_TEST)[:PCA_SAMPLE_SIZE]
projected = pca.transform(X_test[plot_rows])

# Cùng bảng màu với các notebook mô hình; mã cụm mang nghĩa riêng trong từng phương pháp.
cluster_colors = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
]
fig, axes = plt.subplots(1, 3, figsize=(18, 6.5), sharex=True, sharey=True)
for ax, method in zip(axes, METHODS):
    info = results[method]["metrics"]
    plot_labels = test_labels[method].to_numpy()[plot_rows]
    for cluster in range(info["best_k"]):
        selected = plot_labels == cluster
        ax.scatter(projected[selected, 0], projected[selected, 1],
                   color=cluster_colors[cluster], s=10, alpha=0.5, linewidths=0, label=f"Cụm {cluster}")
    ax.set(title=f"{METHOD_NAMES[method]}\nK={info['best_k']}, seed={info['best_seed']}",
           xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.1%} phương sai train)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17),
              ncol=min(info["best_k"], 5), frameon=False, markerscale=2)
    ax.grid(alpha=0.2)
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} phương sai train)")
fig.suptitle(f"So sánh PCA — cùng {len(plot_rows):,}/{N_TEST:,} giao dịch test và cùng hệ trục")
fig.text(0.5, 0.02, "Mã cụm và màu chỉ có ý nghĩa trong từng phương pháp.", ha="center", fontsize=10)
fig.tight_layout(rect=(0, 0.07, 1, 0.95))
save_figure(fig, "pca_comparison.png")
print(f"Hai trục PCA giữ {pca.explained_variance_ratio_.sum():.2%} phương sai train.")


## 10. Tra cứu giao dịch theo cụm

Đổi `LOOKUP_METHOD`, `LOOKUP_CLUSTER`, `LOOKUP_SPLIT`, `LOOKUP_LIMIT` ở mục 1,
rồi chạy lại ô cấu hình và ô hiển thị bên dưới. Chọn `all` để xem cả ba tập.

Ghép nhãn với `transaction_lookup.parquet` theo `transaction_id`, sắp theo thời gian.
Số tiền dùng VND gốc; `source_file` và `source_row` cho phép tra ngược Excel.
Mặc định tài khoản chỉ hiện 4 ký tự cuối; đặt `MASK_ACCOUNTS=False` để xem đầy đủ tại máy.

In [ ]:
transaction_lookup = pd.read_parquet(PREPARED_DIR / "transaction_lookup.parquet")


def lookup_cluster(method, cluster, split="test", limit=20, mask_accounts=True):
    """Tra cứu một cụm; tổng số giao dịch được lưu trong attrs['total_matches']."""
    assert method in METHODS and split in [*SPLITS, "all"], "Phương pháp hoặc tập không hợp lệ."
    assert 0 <= cluster < results[method]["metrics"]["best_k"] and limit > 0
    labels = results[method]["labels"]
    selected = labels[labels["cluster"] == cluster]
    if split != "all":
        selected = selected[selected["split"] == split]
    joined = selected.merge(transaction_lookup, on="transaction_id", how="left", validate="one_to_one")
    assert joined["transaction_datetime"].notna().all(), "Thiếu thông tin tra cứu giao dịch."
    view = joined.sort_values(["transaction_datetime", "transaction_id"]).head(limit).copy()
    if mask_accounts:
        for column in ["accountsourcenumber", "accountdestnumber"]:
            view[column] = view[column].map(
                lambda v: "*" * max(0, len(str(v)) - 4) + str(v)[-4:] if pd.notna(v) else v
            )
    columns = ["transaction_id", "split", "cluster", "transaction_datetime", "transactionamount",
               "accountsourcenumber", "accountdestnumber", "referencenumber", "source_file", "source_row"]
    view = view[columns].reset_index(drop=True)
    view.attrs["total_matches"] = len(joined)
    return view


In [ ]:
transactions = lookup_cluster(LOOKUP_METHOD, LOOKUP_CLUSTER, LOOKUP_SPLIT, LOOKUP_LIMIT, MASK_ACCOUNTS)
print(f"{METHOD_NAMES[LOOKUP_METHOD]} — cụm {LOOKUP_CLUSTER} — tập {LOOKUP_SPLIT}: "
      f"{transactions.attrs['total_matches']:,} giao dịch; hiển thị {len(transactions)} dòng.")
display(transactions.style.format({"transactionamount": "{:,.0f} VND"}, na_rep="—"))


## 11. Nhận xét và giới hạn

Nhận xét lấy từ các bảng vừa tính, nên tự cập nhật khi chạy lại.

In [ ]:
final_scores = comparison_final.set_index("method")
best_sil = final_scores["silhouette"].idxmax()
best_db = final_scores["davies_bouldin"].idxmin()
winners = ranking.loc[ranking["overall_rank"] == 1, "method"].map(METHOD_NAMES).tolist()
reference_method = final_scores["validation_silhouette_mean"].idxmax()
reference_score = final_scores.loc[reference_method]
closest = method_agreement.loc[method_agreement["ari"].idxmax()]
smallest = cluster_sizes.loc[cluster_sizes["transaction_count"].idxmin()]

notes = [
    f"**Mô hình cuối trên test:** {METHOD_NAMES[best_sil]} có Silhouette cao nhất "
    f"({final_scores.loc[best_sil, 'silhouette']:.6f}); {METHOD_NAMES[best_db]} có Davies–Bouldin thấp nhất "
    f"({final_scores.loc[best_db, 'davies_bouldin']:.4f}).",
    f"**Hạng tổng hợp tốt nhất:** {' và '.join(winners)}; hạng trung bình {ranking['mean_rank'].min():.3f} "
    f"trên {len(ranking_metrics)} chỉ số, trọng số bằng nhau.",
    f"**Mô hình tham chiếu theo validation:** {METHOD_NAMES[reference_method]}, Silhouette trung bình "
    f"{reference_score['validation_silhouette_mean']:.6f} ± {reference_score['validation_silhouette_std']:.6f}.",
    f"**Cặp có ARI cao nhất:** {METHOD_NAMES[closest['method_a']]} và {METHOD_NAMES[closest['method_b']]} "
    f"(ARI={closest['ari']:.4f}, NMI={closest['nmi']:.4f}).",
    f"**Cụm nhỏ nhất:** {METHOD_NAMES[smallest['method']]}, mã {int(smallest['cluster'])}: "
    f"{int(smallest['transaction_count']):,} giao dịch ({smallest['transaction_share']:.2%}).",
]
display(Markdown("\n\n".join(f"- {note}" for note in notes)))


**Giới hạn:** dữ liệu giả lập không có nhãn đúng; các chỉ số không phải accuracy hay bằng chứng gian lận.
ARI/NMI đo mức đồng thuận, không quyết định mô hình tốt nhất. Xếp hạng là tóm tắt mô tả,
chưa có kiểm định để khẳng định ưu thế thống kê; dải theo seed là độ lệch chuẩn, không phải khoảng tin cậy.
Calinski–Harabasz chỉ được xếp hạng khi cùng K. Kết quả test dùng để báo cáo, không chọn lại K/seed.

## 12. Lưu kết quả tổng hợp

Trong `output/comparison/`:
- `comparison_final.csv`: chỉ số mô hình cuối, K/seed, thông tin validation và xếp hạng.
- `comparison_by_k.csv`: trung bình, độ lệch chuẩn và số seed trên test cho từng phương pháp/K.
- `method_agreement.csv`: ARI/NMI giữa ba cặp phương pháp.
- `figures/`: ba hình so sánh cũ và hình mới **`pca_comparison.png`**.

CSV giữ độ chính xác gốc; chỉ làm tròn khi hiển thị. Chạy lại cập nhật cùng thư mục.

In [ ]:
exports = {
    "comparison_final.csv": comparison_final,
    "comparison_by_k.csv": comparison_by_k,
    "method_agreement.csv": method_agreement,
}
for filename, frame in exports.items():
    frame.to_csv(COMPARISON_DIR / filename, index=False, encoding="utf-8-sig")
print("Đã lưu vào:", COMPARISON_DIR)
print("3 CSV và 4 biểu đồ; PCA nằm trong figures/pca_comparison.png.")
